# NB12 — Visualisasi Sebaran Cluster & Noise (Hasil Final)

**Tujuan:** menjawab masukan penguji seminar hasil — menampilkan sebaran titik data
hasil clustering (terkelompok vs noise) secara visual, dan memastikan foto asli di
balik tiap titik noise bisa ditelusuri.

**PENTING — notebook ini HANYA MEMBACA output yang sudah ada.** Tidak ada pipeline
NB01/NB05/NB10/NB11 yang dijalankan ulang, dan tidak ada apa pun yang ditulis ke
`output_nb01/`, `output_nb05/`, `output_nb10/`, atau `output_nb11/`. Semua output
notebook ini masuk ke folder baru `output_nb12/` agar seluruh angka Bab IV
(15.248 wajah, DBCV 0,6588, 139 klaster, dst.) tidak tersentuh.

**Input:**
- `output_nb11/skenario_e_results.pkl` → `labels_E` (label final, 14.790 wajah inlier)
  dan `meta` (metadata paralel: `photo_path`, `bbox`)
- `output_nb10/umap_coords_inlier.npy` → `X_umap` (koordinat 30-dim, ruang tempat
  HDBSCAN benar-benar dijalankan)

**Output:** 4 gambar PNG ke `output_nb12/`.

**Catatan metodologis wajib dibaca sebelum menafsirkan gambar di bawah:** proyeksi
2D pada notebook ini HANYA untuk visualisasi — bukan dasar penentuan cluster. Warna
tiap titik berasal dari label yang sudah dihitung sebelumnya di ruang 30-dim asli
(`labels_E`, hasil Two-Pass HDBSCAN + GLOSH + Approximate Predict). Jarak antar
klaster dan kepadatan visual di plot 2D TIDAK boleh dibaca sebagai ukuran kuantitatif
kualitas clustering — itu artefak proyeksi (UMAP tidak menjaga jarak global maupun
kepadatan relatif, hanya ketetanggaan lokal). Bukti kuantitatif kualitas clustering
adalah metrik di Tabel IV.6 (DBCV, Silhouette, DBI), dihitung di ruang 30-dim asli,
bukan dari plot ini.

## 0. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import pickle
from pathlib import Path
from collections import OrderedDict

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import cv2
import umap
import hdbscan

from pillow_heif import register_heif_opener
register_heif_opener()

print('Setup OK.')

## 1. Muat output NB10 & NB11 (baca saja)

In [ ]:
BASE       = Path("/content/drive/MyDrive/OTW S.KOM/Embeddings")
NB10_DIR   = BASE / "output_nb10"
NB11_DIR   = BASE / "output_nb11"
OUTPUT_DIR = BASE / "output_nb12"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

X_umap = np.load(NB10_DIR / "umap_coords_inlier.npy")

with open(NB11_DIR / "skenario_e_results.pkl", "rb") as f:
    results_E = pickle.load(f)

labels_E = results_E["labels_E"]
meta     = results_E["meta"]          # paralel dgn labels_E & X_umap

assert len(labels_E) == len(meta) == X_umap.shape[0], "Indeks tidak selaras!"

n_noise_E    = int((labels_E == -1).sum())
n_cluster_E  = len(set(labels_E)) - (1 if -1 in labels_E else 0)

print(f"X_umap        : {X_umap.shape}")
print(f"labels_E      : {len(labels_E):,} titik")
print(f"Jumlah cluster: {n_cluster_E}")
print(f"Noise (S1)    : {n_noise_E}")
print()
print("Kontrol silang (harus cocok Tabel IV.6): 139 klaster, 60 noise")

## 2. Proyeksi 2D (visualisasi saja)

Proyeksi dijalankan dari `X_umap` (30-dim, ruang tempat HDBSCAN dijalankan) — BUKAN
dari embedding 512-dim mentah seperti pada NB03/NB05. Alasannya: ini proyeksi paling
langsung dari ruang yang sama persis dilihat HDBSCAN, sehingga tata letak 2D yang
dihasilkan paling relevan menggambarkan struktur yang benar-benar dipakai algoritma
untuk membentuk cluster.

In [ ]:
reducer_viz = umap.UMAP(
    n_components=2,
    random_state=42,
    n_jobs=1,        # n_jobs=1 wajib berpasangan dgn random_state utk reproducibility
    verbose=True,
)
emb_2d = reducer_viz.fit_transform(X_umap)
print(f"emb_2d shape: {emb_2d.shape}")

## 3. Gambar 1 — Sebaran Final: Cluster vs Noise

In [ ]:
mask_noise = labels_E == -1
mask_clust = labels_E >= 0
unique_labels = sorted(set(labels_E[mask_clust].tolist()))
cmap = plt.colormaps.get_cmap('tab20')

fig, ax = plt.subplots(figsize=(12, 10))

# Cluster — digambar dulu (zorder rendah = latar belakang)
for i, lbl in enumerate(unique_labels):
    m = labels_E == lbl
    ax.scatter(emb_2d[m, 0], emb_2d[m, 1], c=[cmap(i % 20)],
               s=3, alpha=0.6, zorder=1, rasterized=True)

# Noise — digambar terakhir, di atas (zorder tinggi), lebih besar & merah
ax.scatter(emb_2d[mask_noise, 0], emb_2d[mask_noise, 1],
           c='#E53935', s=12, alpha=0.9, zorder=3,
           edgecolors='black', linewidths=0.3,
           label=f'Noise ({n_noise_E})')

ax.set_title(
    f"Sebaran Hasil Clustering Final (Skenario E)\n"
    f"{n_cluster_E} cluster, {n_noise_E} noise dari {len(labels_E):,} wajah "
    f"(proyeksi UMAP 30\u21922 dim, hanya utk visualisasi)",
    fontsize=12, fontweight='bold'
)
ax.set_xlabel("UMAP-1"); ax.set_ylabel("UMAP-2")
ax.legend(markerscale=2, fontsize=10, loc='best')
ax.grid(alpha=0.15)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "gambar1_sebaran_final.png", dpi=150, bbox_inches='tight')
plt.show()

## 4. Gambar 2 — Sebelum vs Sesudah Refinement (S0 vs S1)

Pass-1 (S0) dihitung ulang di sini dengan konfigurasi yang identik dengan cell
konfigurasi `nb11_lof2.ipynb` (`MCS_PASS1=15, MS=10`) — pemenang grid search
`NB10b_Grid_mcs_ms_UMAP30.ipynb` (DBCV=0,8391). Karena HDBSCAN deterministik untuk
input dan parameter yang sama, hasil ini WAJIB identik dengan hasil pass-1 yang
sudah tercatat (123 cluster, 450 noise) — sel di bawah memverifikasi ini secara
eksplisit sebelum plot dibuat.

In [ ]:
clusterer_pass1 = hdbscan.HDBSCAN(
    min_cluster_size=15,
    min_samples=10,
    metric='euclidean',
    cluster_selection_method='eom',
    prediction_data=True,
)
labels_S0 = clusterer_pass1.fit_predict(X_umap)

n_noise_S0   = int((labels_S0 == -1).sum())
n_cluster_S0 = len(set(labels_S0)) - (1 if -1 in labels_S0 else 0)

print(f"S0 (pass-1 saja) : {n_cluster_S0} cluster, {n_noise_S0} noise")
print(f"S1 (final)        : {n_cluster_E} cluster, {n_noise_E} noise")

assert n_cluster_S0 == 123 and n_noise_S0 == 450, (
    "Hasil pass-1 TIDAK cocok dgn catatan NB10b (123 cluster/450 noise) — "
    "kemungkinan X_umap yang dimuat berbeda dari yang dipakai nb11_lof2. "
    "JANGAN lanjut sebelum ini diselidiki."
)
print("\u2713 Cocok dgn baris pemenang NB10b — X_umap terverifikasi konsisten.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

for ax, labels_x, title, n_c, n_n in [
    (axes[0], labels_S0, "S0 \u2014 Pass-1 saja (sebelum refinement)", n_cluster_S0, n_noise_S0),
    (axes[1], labels_E,  "S1 \u2014 Final (Two-Pass + GLOSH + Approx. Predict)", n_cluster_E, n_noise_E),
]:
    m_noise = labels_x == -1
    m_clust = labels_x >= 0
    uniq = sorted(set(labels_x[m_clust].tolist()))
    for i, lbl in enumerate(uniq):
        m = labels_x == lbl
        ax.scatter(emb_2d[m, 0], emb_2d[m, 1], c=[cmap(i % 20)],
                   s=3, alpha=0.6, zorder=1, rasterized=True)
    ax.scatter(emb_2d[m_noise, 0], emb_2d[m_noise, 1],
               c='#E53935', s=10, alpha=0.9, zorder=3,
               edgecolors='black', linewidths=0.3)
    ax.set_title(f"{title}\n{n_c} cluster, {n_n} noise ({n_n/len(labels_x)*100:.1f}%)",
                 fontsize=11, fontweight='bold')
    ax.set_xlabel("UMAP-1"); ax.set_ylabel("UMAP-2")
    ax.grid(alpha=0.15)

fig.suptitle("Efek Refinement Terhadap Noise (kontribusi utama penelitian)",
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "gambar2_sebelum_sesudah_refinement.png", dpi=150, bbox_inches='tight')
plt.show()

## 5. Helper baca foto & crop wajah

Dipakai ulang dari `LABEL_Cluster_Verification.ipynb` (repo `fotoQu-faceEnhancement`)
— satu-satunya implementasi di kedua repo yang menangani RAW kamera dan HEIC iPhone,
bukan hanya JPG. Dua perbaikan diterapkan dibanding versi asal:
1. `render_faces` di sini memakai `set_xticks([])`/`set_yticks([])`, BUKAN
   `ax.axis("off")` — versi asal memanggil `axis("off")` untuk semua axis sebelum
   loop, sehingga border highlight tidak pernah tampil.
2. Tidak ada `cv2.cvtColor(..., COLOR_BGR2RGB)` ganda pada keluaran `thumb()` —
   `thumb()` sudah mengembalikan RGB; versi `show_review_gallery` di notebook asal
   mengonversi ulang sehingga wajah tampak kebiruan. Di sini tidak diulang.

In [ ]:
_PHOTO_CACHE = OrderedDict()
_CACHE_CAP = 64
RAW_EXTS = {'.raf', '.cr2', '.nef', '.arw', '.dng', '.rw2', '.orf'}

def imread_any(p):
    if p in _PHOTO_CACHE:
        _PHOTO_CACHE.move_to_end(p)
        return _PHOTO_CACHE[p]
    img = cv2.imread(p)
    if img is None:
        ext = Path(p).suffix.lower()
        try:
            if ext in RAW_EXTS:
                import rawpy
                with rawpy.imread(p) as raw:
                    img = cv2.cvtColor(raw.postprocess(), cv2.COLOR_RGB2BGR)
            else:
                from PIL import Image
                img = cv2.cvtColor(np.array(Image.open(p).convert("RGB")), cv2.COLOR_RGB2BGR)
        except Exception:
            img = None
    _PHOTO_CACHE[p] = img
    if len(_PHOTO_CACHE) > _CACHE_CAP:
        _PHOTO_CACHE.popitem(last=False)
    return img

def thumb(gi, size=112, padding=0.20):
    m = meta[gi]
    img = imread_any(m["photo_path"])
    if img is None:
        return np.zeros((size, size, 3), np.uint8)
    H, W = img.shape[:2]
    x1, y1, x2, y2 = m["bbox"]
    bw, bh = x2 - x1, y2 - y1
    x1 = int(max(0, x1 - padding * bw)); y1 = int(max(0, y1 - padding * bh))
    x2 = int(min(W, x2 + padding * bw)); y2 = int(min(H, y2 + padding * bh))
    crop = img[y1:y2, x1:x2]
    if crop.size == 0:
        return np.zeros((size, size, 3), np.uint8)
    return cv2.cvtColor(cv2.resize(crop, (size, size)), cv2.COLOR_BGR2RGB)

def render_faces(idx_list, title="", ranks=None, cols=8):
    n = len(idx_list)
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 1.5, rows * 1.7))
    axes = np.array(axes).reshape(-1)
    for j, ax in enumerate(axes):
        ax.set_xticks([]); ax.set_yticks([])
        if j < n:
            gi = idx_list[j]
            ax.imshow(thumb(gi))
            lbl = str(ranks[j]) if ranks is not None else str(j)
            ax.set_title(lbl, fontsize=7)
        else:
            ax.set_visible(False)
    fig.suptitle(title, fontsize=13, fontweight='bold', y=1.01)
    plt.tight_layout()
    return fig

print("Helper siap.")

## 6. Gambar 3 — Contact Sheet Seluruh Wajah Noise

Setiap ubin diberi nomor urut (0 sampai N-1). Nomor yang sama dipakai di Gambar 4
untuk menandai posisi titik itu di ruang 2D — pasangan Gambar 3+4 inilah yang
menjawab permintaan penguji: "kita bisa memastikan fotonya".

In [ ]:
noise_idx = np.where(labels_E == -1)[0]
noise_order = np.sort(noise_idx)   # urutan tetap & reproducible
ranks = list(range(len(noise_order)))

print(f"Total wajah noise: {len(noise_order)}")

fig3 = render_faces(
    noise_order.tolist(), ranks=ranks, cols=8,
    title=f"Seluruh Wajah Noise Final (n={len(noise_order)}) \u2014 nomor cocok dgn Gambar 4"
)
fig3.savefig(OUTPUT_DIR / "gambar3_contact_sheet_noise.png", dpi=150, bbox_inches='tight')
plt.show()

## 7. Gambar 4 — Sebaran dengan Anotasi Nomor (pasangan Gambar 3)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 12))

for i, lbl in enumerate(unique_labels):
    m = labels_E == lbl
    ax.scatter(emb_2d[m, 0], emb_2d[m, 1], c=[cmap(i % 20)],
               s=3, alpha=0.5, zorder=1, rasterized=True)

ax.scatter(emb_2d[noise_order, 0], emb_2d[noise_order, 1],
           c='#E53935', s=14, alpha=0.95, zorder=3,
           edgecolors='black', linewidths=0.4)

for rank, gi in zip(ranks, noise_order):
    ax.annotate(str(rank), (emb_2d[gi, 0], emb_2d[gi, 1]),
                fontsize=5.5, color='black', zorder=4,
                xytext=(2, 2), textcoords='offset points')

ax.set_title(
    f"Sebaran dgn Nomor Wajah Noise (n={len(noise_order)})\n"
    f"Cocokkan nomor titik di sini dgn nomor ubin di Gambar 3 utk melihat foto aslinya",
    fontsize=12, fontweight='bold'
)
ax.set_xlabel("UMAP-1"); ax.set_ylabel("UMAP-2")
ax.grid(alpha=0.15)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "gambar4_sebaran_beranotasi.png", dpi=150, bbox_inches='tight')
plt.show()

print("Catatan: pada wilayah noise yang rapat, sebagian nomor bisa bertumpuk secara "
      "visual di layar kecil \u2014 perbesar PNG tersimpan (300%+) saat dipakai di sidang.")

## 8. Ringkasan

4 gambar tersimpan ke `output_nb12/`:
1. `gambar1_sebaran_final.png` — sebaran hasil final, cluster vs noise
2. `gambar2_sebelum_sesudah_refinement.png` — S0 (pass-1) vs S1 (final), efek kontribusi utama
3. `gambar3_contact_sheet_noise.png` — seluruh wajah noise, bernomor
4. `gambar4_sebaran_beranotasi.png` — sebaran dgn nomor yang sama, utk ditelusuri balik ke Gambar 3

**Untuk sidang:** kalau penguji bertanya soal titik/wajah tertentu, tunjuk titik di
Gambar 4, baca nomornya, cari nomor yang sama di Gambar 3 \u2014 foto aslinya
langsung terlihat. Ingatkan bahwa jarak & kepadatan di plot ini ilustratif (artefak
proyeksi UMAP 2D), bukan bukti kuantitatif \u2014 bukti kuantitatif ada di Tabel IV.6.

In [ ]:
print("=" * 60)
print("  RINGKASAN NB12")
print("=" * 60)
print(f"  Total wajah (inlier)  : {len(labels_E):,}")
print(f"  Cluster (S1, final)   : {n_cluster_E}")
print(f"  Noise (S1, final)     : {n_noise_E}")
print(f"  Noise (S0, pass-1)    : {n_noise_S0}")
print(f"  Output disimpan ke    : {OUTPUT_DIR}")
print("=" * 60)